In [ ]:
# colab 환경에서 학습을 진행하실 분들은 구글드라이브를 연동해주세요
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# !pip install vctube
# !pip install youtube_dl
# !pip install pydub
# !pip install youtube_transcript_api

### install / import

In [ ]:
!pip install webdriver_manager
!pip install selenium

In [6]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import requests
from bs4 import BeautifulSoup
import html
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

In [12]:
def get_url(sample): 
    
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
    
    result = []
    for music in sample:
        try:
            title = music[1]
            artist = music[2]
            url = f'https://www.youtube.com/results?search_query={title}+{artist}+audio'
            driver.get(url)
            
            # 요소가 로드될 때까지 최대 10초간 대기
            first_result = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "a#video-title"))
            )
            
            first_video_url = first_result.get_attribute('href')
            result.append([music[0], title, artist, music[3], first_video_url])
        except Exception as e:
            print(f"Error processing {music}")
            result.append([music[0], title, artist, music[3], None])
    driver.quit()
    return result


In [8]:
def MatchPlayerRecords(musicdf, batchsize):
    max_workers = 6  # 사용 가능한 최대 CPU 코어 수
    
    batches = [musicdf[i:i + batchsize] for i in range(0, len(musicdf), batchsize)]

    new_result = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # 각 matchid에 대해 extractPlayerRecords 함수를 병렬로 실행
        future_to_batch = {executor.submit(get_url, batch): batch for batch in batches}
        
        for i, future in enumerate(as_completed(future_to_batch), 1):
            result = future.result()
            if result:
                new_result = new_result + result

            if i % (len(batches) // max_workers) == 0: 
                print(f"Completed batch {i} of {len(batches)}")

    return new_result

In [9]:
# test 

metadata = pd.read_csv('metadata.csv', index_col=False)
sample = [list(row) for row in metadata.values][0:12]
batchsize = round(len(sample) / 6)
result = MatchPlayerRecords(sample, batchsize)

Completed batch 1 of 6
Completed batch 2 of 6
Completed batch 3 of 6
Completed batch 4 of 6
Completed batch 5 of 6
Completed batch 6 of 6


In [ ]:
inputlist = [list(row) for row in metadata.values][0:600]
batchsize = round(len(inputlist) / 6) 
result1 = MatchPlayerRecords(inputlist, batchsize)

In [27]:
inputlist = [list(row) for row in metadata.values][600:]
batchsize = round(len(inputlist) / 6) 
result2 = MatchPlayerRecords(inputlist, batchsize)

Error processing [624, '니 사진만 쳐다보는 일', '순순희', 'Indie']
Error processing [641, '#첫사랑', '볼빨간사춘기', 'Indie']
Error processing [715, '잘 지내', '적재', 'Blues']
Error processing [1769, 'Rodeo (Remix)', 'Flo Milli', 'Hiphop_Overseas']
Error processing [2027, '어땠을까 (Feat. 박정현)', '싸이 (PSY)', 'Hiphop']
Error processing [738, '널 보고 있으면', '고막남친단 (테종, 홍이삭)', 'Blues']
Error processing [747, '사랑하는 사람들에게 가장 상처 주는 키를 우리는 모두 가지고 있어', '김사월', 'Blues']
Error processing [1087, '벚꽃 엔딩', '버스커 버스커', 'Rock']
Error processing [800, 'Oh Friday (루디고 GO X 송예빈)', '송예빈', 'Blues']
Error processing [2139, 'Je Ne Sais Quoi', 'NCT 127', 'Hiphop']
Completed batch 1 of 6
Error processing [1456, 'Last Night On Earth', 'Green Day', 'Rock_Overseas']
Completed batch 2 of 6
Completed batch 3 of 6
Completed batch 4 of 6
Completed batch 5 of 6
Completed batch 6 of 6


In [28]:
result = result1 + result2

metadata_url = pd.DataFrame(result, columns=['Id', 'Title', 'Artist', 'Genre', 'Url'])
metadata_url.sort_values(by='Id', inplace=True)
metadata_url= metadata_url.reset_index(drop=True)

In [30]:
metadata_url = metadata_url.reset_index(drop=True)
metadata_url['Id'] = metadata_url.index
metadata_url = metadata_url[['Id', 'Title', 'Artist', 'Genre', 'Url']]

In [33]:
metadata_url.to_csv('metadata_url.csv', index=False)